# Hallmark demo: command line

Hallmark keeps track of the data files in a science project.

For every file it records the path, a checksum (a code that changes when the file changes),
and the values written in the file name
(for example the spin `a` and angle `i` in `a0.5_i30.h5`).
This record is called the catalog. It lives in a small folder called `.hm`.

The `.hm` folder is a Git repository. You can share it on GitHub like code,
while the large data files stay on your disk or on a data server.

This notebook uses the `hm` command. For the same steps in Python, see `demo_python.ipynb`.

This notebook shows:

1. Track files on your computer: `init`, `info`, `add`, `status`, `commit`, `log`, `checkout`, `branch`, `set-config`
2. Record files that are on a server, then download them: `add URL`, `download`
3. Share a catalog: `clone`

This notebook uses the Bash kernel (`pip install bash_kernel`, then `python -m bash_kernel.install`).
Run the cells from top to bottom.

## 0. Setup

We work in a new temporary folder, so the demo does not leave files in the repository.

In [ ]:
WORK="$(mktemp -d)"
cd "$WORK"
pwd

## Part 1. Track files on your computer

### 1.1 Create a repository

`hm init` creates a folder with an empty `.hm` inside.
It does not look at your files or download anything.

In [ ]:
hm init my-data
ls -a my-data

### 1.2 See where the repository is

`hm info` prints the location of the `.hm` folder and of your files.

In [ ]:
cd my-data
hm info

### 1.3 Make some data files

In these names, `a` is the spin and `i` is the angle.

In [ ]:
for a in 0 0.5; do
  for i in 30 60; do
    echo "spin $a, angle $i" > "a${a}_i${i}.h5"
  done
done
ls

### 1.4 Add the files

Tell Hallmark the pattern of the file names.
Each name in `{ }` becomes a column in the catalog.

In [ ]:
hm add "a{a}_i{i}.h5"

The catalog is the file `.hm/data.tsv`.
There is one row per file, with its checksum (`sha1`) and the values from its name.

In [ ]:
cat .hm/data.tsv

### 1.5 Check the status

`hm status` shows what will be saved by the next commit.

In [ ]:
hm status

### 1.6 Commit

`hm commit` saves this version of the catalog.
A copy of each file is stored in `.hm/objects`, so you can go back to it later.
`hm log` lists the commits, newest first.

In [ ]:
hm commit -m "First dataset"
hm log

### 1.7 Change a file and commit again

After you edit files, run `hm add .` to update the catalog with the pattern you already set.
`hm status` then shows the file as modified.

In [ ]:
echo "spin 0, angle 30, fixed" > a0_i30.h5
hm add .
hm status
hm commit -m "Fix a0_i30"

### 1.8 Try something on a branch

`hm checkout NAME` switches to a branch. If the branch does not exist yet, it is created.
Use a branch to try changes without touching `main`.

Here we switch to a new branch, delete two files there, and commit.

In [ ]:
hm checkout only-spin-0
rm a0.5_*.h5
hm add .
hm commit -m "Keep only spin 0"
ls

`hm branch` lists the branches. The `*` marks the one you are on.

When we switch back to `main`, Hallmark puts back the files that `main` has.

In [ ]:
hm branch
hm checkout main
ls

### 1.9 Change the file pattern

If new files use a different naming pattern, set it with `hm set-config --fmt`,
then run `hm add .`. Stage the removal of the old files with `hm add .` first.
We do this on its own branch.

In [ ]:
hm checkout new-names
rm *.h5
hm add .
for i in 30 60; do echo "model b, angle $i" > "b1_i${i}.h5"; done
hm set-config --fmt "b{b}_i{i}.h5"
hm add .
hm commit -m "Switch to b files"
cat .hm/data.tsv
hm checkout main
cd ..

### 1.10 Negative numbers in file names

Some file names write a minus sign as `m`. For example `am0.5` means a spin of `-0.5`.
Set a rule for this with `hm set-config --encoding`, then run `hm add --regex`.

Note: right now this rule only works for a field named `aspin`.

In [ ]:
hm init spins
cd spins
for f in am0.5_i30.h5 a0_i30.h5 a0.5_i30.h5; do echo "$f" > "$f"; done
hm set-config --fmt "a{aspin}_i{i}.h5" --encoding "aspin=m([0-9.]+)"
hm add --regex "a{aspin}_i{i}.h5"
cat .hm/data.tsv
cd ..

## Part 2. Data on a server

### 2.1 Connect to the demo server

The demo server has four test files:
`M87_001.txt`, `M87_002.txt`, `SgrA_001.txt` and `SgrA_002.txt`.
Hallmark gets them over SSH.

Do this once. Add these lines to `~/.ssh/config`:

```
Host hallmark-demo
    HostName <server-address>
    User <your-username>
    IdentityFile ~/.ssh/<your-key-file>
```

Use your own username and key. Keep the name `hallmark-demo`, because the notebook uses it.

Then run `ssh hallmark-demo` in a terminal, type `yes`, then type `exit`.

The next cell shows your SSH version. It must be 9.6 or newer.

In [ ]:
ssh -V

In [ ]:
DATA_URL="ssh://hallmark-demo/srv/hallmark_demo_data"
echo "Using $DATA_URL"

### 2.2 Record the files on the server

`hm add` also accepts a URL with a file name pattern.
Hallmark looks at the server and records each file that matches.
Nothing is downloaded yet, only names and checksums.

In [ ]:
hm init eht-data
cd eht-data
hm add "$DATA_URL/{src}_{day:03d}.txt" 2>/dev/null
cat .hm/data.tsv
hm commit -m "Add archive catalog"

### 2.3 Preview a download

`--dry-run` shows what would be downloaded, without downloading or asking.

In [ ]:
hm download --all --dry-run

### 2.4 Download some files

`--filter` picks files by name. `*` matches any characters.
Hallmark shows the plan and asks `[y/N]`. Here we answer `y` with `echo y`.
Each file is checked against its checksum.

In [ ]:
echo y | hm download --filter "SgrA_*" 2>/dev/null
ls SgrA_*.txt

### 2.5 Download everything

In [ ]:
echo y | hm download --all 2>/dev/null
ls *.txt
cd ..

## Part 3. Share a catalog

`hm clone` copies a Hallmark repository and its history.
In real use the source is a Git URL, for example `hm clone git@github.com:team/eht-data.git`.
Here we clone the local repository from Part 2.

### 3.1 Catalog only

`--no-download` copies the catalog and history, but no data files.

In [ ]:
hm clone eht-data/.hm catalog-only --no-download
ls -a catalog-only

### 3.2 Catalog and data

Without `--no-download`, clone shows how many files it will download and asks first.

In [ ]:
echo y | hm clone eht-data/.hm full-copy 2>/dev/null
ls full-copy

### 3.3 Saying no

If you answer `n` (or just press Enter), you keep the catalog and can download later.

In [ ]:
echo n | hm clone eht-data/.hm later-copy
cd later-copy
echo y | hm download --filter "M87_*" 2>/dev/null
cd ..

## Clean up

Delete the temporary folder.

In [ ]:
cd
rm -rf "$WORK"

## Command summary

| Command | What it does |
|---|---|
| `hm init DIR` | Make a new, empty repository |
| `hm info` | Show where the repository is |
| `hm add "PATTERN"` | Add local files that match a name pattern |
| `hm add .` | Update the catalog with the current pattern |
| `hm add --regex "PATTERN"` | Add files using the encoding rules in the config |
| `hm add "URL/PATTERN"` | Record files on a server, without downloading |
| `hm status` | Show what the next commit will save |
| `hm commit -m "MESSAGE"` | Save this version |
| `hm log` | List commits |
| `hm branch` | List branches |
| `hm checkout NAME` | Switch to a branch, or create it |
| `hm set-config --fmt "PATTERN"` | Change the file name pattern |
| `hm set-config --encoding "FIELD=REGEX"` | Add an encoding rule |
| `hm download --all` / `--filter` / `PATHS` | Download files in the catalog (asks first) |
| `hm download --dry-run` | Show what would be downloaded |
| `hm clone SOURCE DIR` | Copy a repository and download its data (asks first) |
| `hm clone SOURCE DIR --no-download` | Copy only the catalog and history |